In [7]:
import boto3
import json
bedrock = boto3.client(service_name = "bedrock-runtime",region_name= "us-east-1")
bedrock_agent = boto3.client(service_name = "bedrock-agent", region_name = "us-east-1")

In [8]:
MODEL_ID = "amazon.nova-micro-v1:0"

In [9]:
temperature = 0.7
inference_config = {"temperature":temperature}
system_prompts = [{"text": "You are a virtual travel assistant that suggests destinations based on user preferences."
                + "Only return destination names and a brief description."}]
messages =[]
messages_1 = {"role":"user","content":[{"text":"Create a list of 3 travel destinations"}]}
messages.append(messages_1)
response = bedrock.converse(modelId = MODEL_ID, messages= messages ,system = system_prompts, inferenceConfig = inference_config)
def print_response(response):
    model_response = response.get('output', {}).get('message', {}).get('content', [{}])[0].get('text', '')

    print("✈️ Your suggested travel destinations:")
    print(model_response)
print_response(response)

✈️ Your suggested travel destinations:
1. **Kyoto, Japan** - Experience traditional Japanese culture, stunning temples, and beautiful gardens.

2. **Santorini, Greece** - Enjoy picturesque sunsets, white-washed buildings, and crystal-clear Aegean Sea views.

3. **Maui, Hawaii, USA** - Discover stunning beaches, lush rainforests, and volcanic landscapes.


In [10]:
message_2 = {"role": "user","content":[{"text": "Only suggest those locations which are no more than one short flight"}]}
messages.append(message_2)
response = bedrock.converse(
    modelId = MODEL_ID,
    system = system_prompts,
    messages = messages,
    inferenceConfig = inference_config
)
print_response(response)

✈️ Your suggested travel destinations:
1. **Miami, Florida**  
   Experience vibrant beaches, lively nightlife, and rich Cuban culture in this tropical paradise just a short flight from many major U.S. cities.

2. **San Diego, California**  
   Discover beautiful coastlines, world-class zoos, and stunning wildlife, all within easy reach by plane from numerous locations.

3. **Charleston, South Carolina**  
   Indulge in historic charm, Southern cuisine, and beautiful gardens, all accessible via a quick flight from various U.S. hubs.


In [11]:
message_3 = {"role":"user","content":[{"text":"Suggest me a place from your previous suggestion which is sunny"}]}
messages.append(message_3)
response = bedrock.converse(
    modelId = MODEL_ID,
    system = system_prompts,
    messages = messages,
    inferenceConfig = inference_config
)
print_response(response)

✈️ Your suggested travel destinations:
1. **Tampa, Florida**
   - A vibrant city known for its beautiful beaches, vibrant nightlife, and rich cultural experiences. 

2. **Orlando, Florida**
   - Famous for its theme parks like Disney World and Universal Studios, offering endless fun for all ages.

3. **Miami, Florida**
   - A bustling metropolis known for its stunning beaches, lively nightlife, and rich Latin American culture.

**Previous sunny suggestion:** **Miami, Florida**
   - Known for its year-round sunshine, beautiful beaches, and vibrant cultural scene.


In [14]:
try:
    response = bedrock_agent.create_prompt(
      name = "Travel-Agent-Prompt",
      description = "Checks if all trip information has been provided",
      variants = [
          {
              "name": "VariantV1",
              "modelId": MODEL_ID,
              "templateType": "CHAT",
              "inferenceConfiguration":{
                  "text":{
                      "temperature":0.4
                  }
              },
              "templateConfiguration":{
                  "chat":{
                      "system":[
                          {
                              "text": """You are a travel agent evaluating trip requests for custom itineraries. 
                                Review the message carefully and answer YES or NO to the following screening questions. 
                                Be strict—if any detail is missing or unclear, answer NO.

                                A) Is the destination clearly stated?
                                B) Are the travel dates within a reasonable range (not last−minute or over a year away)?
                                C) Does the request avoid high−risk or restricted activities (e.g., extreme sports, off−grid travel)?
                                D) Is there any mention of a valid passport or travel documentation?
                                E) Is there enough information to follow up with a proposed itinerary?"""
                          }
                      ],
                      "messages":[
                          {"role":"user","content":[{"text":"Trip request {{event_request}}"}]}
                      ],
                      "inputVariables":[
                          {"name":"event_request"}
                      ]
                  }
              }
          }
      ]
    )
    print("Created!")
    prompt_arn = response['arn']
except bedrock.exceptions.ConflictException as e:
    print("Already exists!")
    response = bedrock.list_prompts()
    prompt = next((prompt for prompt in response['promptSummaries'] if prompt['name'] == "TripBooker_xyz"), None)
    prompt_arn = prompt['arn']
prompt_arn


ConflictException: An error occurred (ConflictException) when calling the CreatePrompt operation: Couldn't perform CreatePrompt operation. The name Travel-Agent-Prompt already exists for id D23NFCVC44. Retry your request with a different name.

In [23]:
response = bedrock.converse(
    modelId = prompt_arn,
    promptVariables = {
        "event_request":{
            "text":"""
            Hi there! I'm planning a trip to Italy with my partner and would love some help organizing the itinerary. We're hoping to travel between September 10–20 this year, ideally flying into Rome and spending a few days in Florence and Venice as well. We’d love recommendations on tours, cultural sites, and good local restaurants. We’re not interested in anything risky like skydiving or hiking remote trails — just want a relaxing and enriching experience. We both have valid passports. Let me know what other details you need!
                """
        }
    }
    
)
print(response["output"]["message"]["content"][0]["text"])

A) YES - The destination is clearly stated as Italy.
B) YES - The travel dates are within a reasonable range (September 10–20 this year).
C) YES - The request avoids high-risk or restricted activities.
D) YES - There is a mention of valid passports.
E) YES - There is enough information to follow up with a proposed itinerary (destination, travel dates, specific cities of interest, and preferences for tours, cultural sites, and restaurants).

Therefore, the answer to whether the request is complete is YES.
